In [2]:
# import necessary dependencies 
import os
import re
import pandas as pd
from collections import defaultdict

In [3]:
# check all of the individual dataset files and verify what the name format is like and the format of the data itself within each 
# file to use when converting to one single pkl file 
FOLDER = "FallAllD"
files = sorted(os.listdir(FOLDER))

print(len(files))
for f in sorted(files)[:15]:
    print(f)

with open(os.path.join(FOLDER, files[0])) as f:
    for _ in range(3):
        print(f.readline())


26420
S01_D1_A013_T01_A.dat
S01_D1_A013_T01_B.dat
S01_D1_A013_T01_G.dat
S01_D1_A013_T01_M.dat
S01_D1_A013_T02_A.dat
S01_D1_A013_T02_B.dat
S01_D1_A013_T02_G.dat
S01_D1_A013_T02_M.dat
S01_D1_A013_T03_A.dat
S01_D1_A013_T03_B.dat
S01_D1_A013_T03_G.dat
S01_D1_A013_T03_M.dat
S01_D1_A013_T04_A.dat
S01_D1_A013_T04_B.dat
S01_D1_A013_T04_G.dat
4155,55,-82

4157,55,-126

4157,50,-175



In [4]:
# Filenames follow the pattern S<SubjectID>_D<Device>_A<ActivityID>_T<TrialNo>_<SensorLetter>.dat
# example: S01_D1_A013_T01_A.dat -> Subject 1, Device 1, Activity 13, Trial 1, Accelerometer
FILE_NAME_RE = re.compile(r"S(\d+)_D(\d+)_A(\d+)_T(\d+)_([ABGM])\.dat$", re.IGNORECASE)
# initialize a sensor map by mapping the single-letter sensor code in the filename to the column name used in the final dataframe
SENSOR_MAP = {"A": "Acc", "G": "Gyr", "M": "Mag", "B": "Bar"}
# parse the file name to determine which subject, activity, device, trial and sensor were used
# returns None if it isn't the same structure
def parse_filename(fname):
    m = FILE_NAME_RE.match(fname)
    if not m:
        return None
    sub, dev, act, trial, sensor = m.groups()
    return {
        "SubjectID": int(sub),
        "Device": int(dev),
        "ActivityID": int(act),
        "TrialNo": int(trial),
        "SensorType": SENSOR_MAP[sensor.upper()],
    }

# build the data frame by parsing each file name and appending the subjectID, Device, ActivityID, TrialNo and SensorType 
# to the dataframe until every file is processed (also groups 4 files split from Acc, Gyr, Mag and Bar to one single row, so that
# the final dataframe only has one row per recording instead of one row per file. 6,605 final rows instead of 26,420 for each file)
def build_dataframe(folder, sep=","):
    groups = defaultdict(dict)
    skipped = []

    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(".dat"):
            continue
        meta = parse_filename(fname)
        if meta is None:
            skipped.append(fname)
            continue
        
        # Group key uniquely identifies one recording across its 4 sensor files
        key = (meta["SubjectID"], meta["Device"], meta["ActivityID"], meta["TrialNo"])
        # header=None because raw files have no header row
        arr = pd.read_csv(os.path.join(folder, fname), sep=sep, header=None).values

        groups[key]["SubjectID"] = meta["SubjectID"]
        groups[key]["Device"] = meta["Device"]
        groups[key]["ActivityID"] = meta["ActivityID"]
        groups[key]["TrialNo"] = meta["TrialNo"]
        groups[key][meta["SensorType"]] = arr

    if skipped:
        print(f"Skipped {len(skipped)} unrecognized files, e.g.: {skipped[:5]}")

    return pd.DataFrame(list(groups.values()))

# construct the data frame separating by commas (determined by running the last code block to see that everything was separated by ",")
df = build_dataframe(FOLDER, sep=",")
print(df.shape)
df.head()

(6605, 8)


,SubjectID,Device,ActivityID,TrialNo,Acc,Bar,Gyr,Mag
0,1,1,13,1,"[[4155, 55, -82], [4157, 55, -126], [4157, 50,...","[[1013.193859563175, 23.71569457530976], [1013...","[[1613, 205, 77], [1626, 202, 74], [1634, 199,...","[[3543, -780, 1995], [3537, -823, 1958], [3520..."
1,1,1,13,2,"[[4002, -50, 156], [4002, -38, 172], [4003, -2...","[[1013.214612231942, 24.11322125434876], [1013...","[[-57, 58, 18], [-64, 57, 15], [-66, 51, 16], ...","[[3701, 401, 1960], [3746, 365, 2026], [3816, ..."
2,1,1,13,3,"[[3983, 40, -335], [3984, 38, -324], [3987, 34...","[[1013.25438772214, 24.49373892784119], [1013....","[[1651, 208, 63], [1652, 203, 60], [1656, 198,...","[[3849, -680, 1685], [3834, -707, 1717], [3857..."
3,1,1,13,4,"[[3959, 165, 197], [3953, 165, 196], [3951, 16...","[[1013.167161107663, 24.72971366882324], [1013...","[[-70, 73, 2], [-67, 80, 4], [-70, 89, 5], [-7...","[[3565, -465, -187], [3543, -468, -176], [3553..."
4,1,1,13,5,"[[3750, -182, 252], [3767, -166, 259], [3781, ...","[[1013.304382379492, 25.49256420612335], [1013...","[[394, -59, 3], [397, -63, -1], [394, -66, -3]...","[[3952, 103, 1828], [3973, 133, 1809], [3972, ..."


In [5]:
# verify the shape (should be roughly 6,605 rows) and check if there are any missing sensor data 
print(df.shape)
df.isna().sum()

(6605, 8)


SubjectID     0
Device        0
ActivityID    0
TrialNo       0
Acc           0
Bar           0
Gyr           0
Mag           0
dtype: int64

In [6]:
# once shape and missing values are verified, convert to pkl file for preprocessing, training and evaluation 
df.to_pickle("FallAllD.pkl")

In [7]:
device_digits = set()
for fname in sorted(os.listdir(FOLDER)):
    m = FILE_NAME_RE.match(fname)
    if m:
        device_digits.add(int(m.group(2)))
print(sorted(device_digits))

[1, 2, 3]
